In [53]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

In [39]:
df = pd.read_csv('bank.csv', sep=";")

df.drop(['default', 'loan'], axis=1, inplace=True)

df.y= (df.y == 'yes').astype(int)

df.columns

Index(['age', 'job', 'marital', 'education', 'balance', 'housing', 'contact',
       'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome',
       'y'],
      dtype='object')

In [11]:
df.isnull().sum()

age          0
job          0
marital      0
education    0
balance      0
housing      0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

#### Question 1
What is the most frequent observation (mode) for the column education?

- unknown
- primary
- secondary X
- tertiary

In [18]:
df.groupby('education')['education'].count().sort_values(ascending=False)

education
secondary    2306
tertiary     1350
primary       678
unknown       187
Name: education, dtype: int64

#### Question 2
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- age and balance
- day and campaign
- day and pdays
- pdays and previous X

In [83]:
numerical = np.array(df.select_dtypes('int').columns[:-1])

categorical = np.array(df.select_dtypes('O').columns)

features = np.concat([numerical, categorical])

df[numerical].corrwith(df.y).sort_values(ascending=False)

duration    0.401118
previous    0.116714
pdays       0.104087
age         0.045092
balance     0.017905
day        -0.011244
campaign   -0.061147
dtype: float64

#### Data Split

In [47]:
df_train, df_test_val = train_test_split(df, test_size=0.4, random_state=42)

df_test, df_val = train_test_split(df_test_val, test_size=0.5, random_state=42)

len(df), len(df_train) , len(df_test), len(df_val)

(4521, 2712, 904, 905)

In [48]:
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

In [50]:
y_train = df_train.y.values
y_test = df_test.y.values
y_val = df_val.y.values

In [51]:
del df_train['y']
del df_test['y']
del df_val['y']

#### Question 3
- Calculate the mutual information score between y and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using round(score, 2).

Which of these variables has the biggest mutual information score?

- contact
- education
- housing
- poutcome X

In [58]:
def mutual_info_y_score(series):
    return mutual_info_score(series, df.y).round(2)

In [59]:
mi = df[categorical].apply(mutual_info_y_score)

mi.sort_values(ascending=False)

poutcome     0.03
month        0.02
job          0.01
contact      0.01
housing      0.01
education    0.00
marital      0.00
dtype: float64

#### Question 4
- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
    - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
    - model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.6
- 0.7
- 0.8
- 0.9 X

In [85]:
def dict_vectorized_fn(df):
    dv = DictVectorizer(sparse=False)
    dicts = df[features].to_dict(orient='records')
    
    X = dv.fit_transform(dicts)
    
    return X


In [112]:
X_train = dict_vectorized_fn(df_train)
X_val = dict_vectorized_fn(df_val)
X_test = dict_vectorized_fn(df_test)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [117]:
y_pred = model.predict_proba(X_val)[:, 1]

term_deposit_decision = (y_pred >= 0.5)

global_model_acc = (term_deposit_decision == y_val).mean()

global_model_acc.round(2)

np.float64(0.91)

#### Question 5
- Let's find the least useful feature using the feature elimination technique.
- Train a model with all these features (using the same parameters as in Q4).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- age
- balance
- marital X
- previous

In [119]:
aux = {}
aux1 = []

for i in range(len(features)):
    
    X_train_reduced = np.delete(X_train, i, axis=1)
    X_val_reduced = np.delete(X_val, i, axis=1)

    model.fit(X_train_reduced, y_train)

    y_pred = model.predict_proba(X_val_reduced)[:, 1]

    term_deposit_decision = (y_pred >= 0.5)

    aux['feature'] = features[i]
    aux['acc'] = abs(global_model_acc - (term_deposit_decision == y_val).mean())

    aux1.append(aux)
    aux = {}

model_acc = pd.DataFrame(aux1)

model_acc.sort_values(by='acc')

,feature,acc
5,pdays,0.001105
8,marital,0.001105
12,month,0.001105
0,age,0.002210
6,previous,0.002210
4,campaign,0.002210
2,day,0.002210
1,balance,0.002210
11,contact,0.002210
9,education,0.002210


#### Question 6
- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter C: [0.01, 0.1, 1, 10, 100].
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these C leads to the best accuracy on the validation set?

- 0.01
- 0.1
- 1
- 10
- 100 X

In [123]:
X_train = dict_vectorized_fn(df_train)
X_val = dict_vectorized_fn(df_val)
X_test = dict_vectorized_fn(df_test)

C = [0.01, 0.1, 1, 10, 100]

for parameter in C:
    model = LogisticRegression(solver='liblinear', C=parameter, max_iter=1000, random_state=42)

    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]

    term_deposit_decision = (y_pred >= 0.5)

    global_model_acc = (term_deposit_decision == y_val).mean()

    print(parameter, global_model_acc.round(3))

0.01 0.897
0.1 0.905
1 0.906
10 0.908
100 0.91
